# Laboratoire 5 — Barren Plateaux et analyse de trainabilité

**Quantum Machine Learning — Master/PhD**

**Objectifs :** Observer et caractériser le phénomène de *barren plateau* dans les circuits quantiques variationnels.

- **Partie A :** Variance du gradient en fonction du nombre de qubits (n=2 à 12)
- **Partie B :** Comparaison ansatz aléatoire vs. structuré (`BasicEntanglerLayers`)
- **Partie C :** Coût global vs. local — différence de variance observée
- **Partie D :** Stratégies d'initialisation : uniforme, normale, identité
- **Partie E :** Visualisation des barren plateaux (boxplots, heatmaps)

**Références :** McClean et al. (2018), *Barren plateaus in quantum neural network training landscapes* ; [SP21] §5.5 ; [VQA26]

In [ ]:
import pennylane as qml
from pennylane import numpy as pnp
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.colors import Normalize
import warnings
warnings.filterwarnings('ignore')

print(f"PennyLane version : {qml.__version__}")
print(f"NumPy version     : {np.__version__}")

---
## Partie A : Variance du gradient en fonction du nombre de qubits

Le phénomène de *barren plateau* (McClean et al., 2018) stipule que pour un circuit quantique aléatoire, la variance du gradient décroît exponentiellement avec le nombre de qubits :

$$\mathrm{Var}\left[\frac{\partial C}{\partial \theta_k}\right] \propto 2^{-n}$$

Nous allons mesurer cette variance pour $n = 2$ à $12$ qubits.

In [ ]:
def compute_gradient_variance(n_qubits, n_layers=2, n_samples=80, seed=42):
    """Calcule la variance du gradient pour un circuit variationnel.
    
    Architecture : AngleEmbedding + BasicEntanglerLayers
    Coût : valeur moyenne de PauliZ sur le premier qubit.
    
    Retourne : variance moyenne des composantes du gradient.
    """
    np.random.seed(seed)
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, diff_method="parameter-shift")
    def circuit(params):
        # params : (n_layers+1, n_qubits), première ligne = embedding
        qml.AngleEmbedding(params[0], wires=range(n_qubits))
        qml.BasicEntanglerLayers(params[1:], wires=range(n_qubits))
        return qml.expval(qml.PauliZ(0))

    all_grads = []
    for _ in range(n_samples):
        params = np.random.uniform(0, 2 * np.pi, (n_layers + 1, n_qubits))
        grad = qml.grad(circuit)(params)
        flat = np.array(grad, dtype=float).flatten()
        all_grads.extend(flat.tolist())

    return float(np.var(all_grads))

In [ ]:
n_qubits_range = range(2, 13)
variances = []

for n in n_qubits_range:
    v = compute_gradient_variance(n, n_layers=2, n_samples=80)
    variances.append(v)
    print(f"n={n:2d}  Var[grad] = {v:.3e}")

# Tracé
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(n_qubits_range), variances, 'o-', color='#E74C3C', lw=2)
ax1.set_xlabel('Nombre de qubits $n$')
ax1.set_ylabel('Var[grad]')
ax1.set_title('Variance du gradient (échelle linéaire)')
ax1.grid(alpha=0.3)

ax2.semilogy(list(n_qubits_range), variances, 's-', color='#2E86C1', lw=2)
ax2.set_xlabel('Nombre de qubits $n$')
ax2.set_ylabel('Var[grad] (log)')
ax2.set_title('Variance du gradient (échelle log) — décroissance exponentielle')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Analyse :** La variance décroît exponentiellement avec $n$, confirmant le barren plateau. En échelle semi-log, la pente donne le taux de décroissance. Au-delà de 8-10 qubits, le gradient devient quasi-nul, rendant l'apprentissage impossible par simple descente de gradient.

---
## Partie B : Ansatz aléatoire vs. structuré

Tous les ansatz ne se valent pas. Un ansatz structuré (ex. `BasicEntanglerLayers`) peut atténuer le barren plateau par rapport à un circuit purement aléatoire.

In [ ]:
def build_structured_ansatz(n_qubits, n_layers):
    """Retourne un circuit structuré avec BasicEntanglerLayers."""
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, diff_method="parameter-shift")
    def circuit(params):
        qml.AngleEmbedding(params[0], wires=range(n_qubits))
        qml.BasicEntanglerLayers(params[1:], wires=range(n_qubits))
        return qml.expval(qml.PauliZ(0))
    return circuit


def build_random_ansatz(n_qubits, n_layers, seed=123):
    """Génère un circuit avec structure aléatoire (gates et connectivité fixes)."""
    np.random.seed(seed)
    dev = qml.device("default.qubit", wires=n_qubits)

    # Pré-générer la structure du circuit
    gate_structure = []
    for l in range(n_layers):
        layer_gates = []
        for i in range(n_qubits):
            gate_type = np.random.choice(['RX', 'RY', 'RZ'])
            layer_gates.append(('single', gate_type, i))
        # Ajouter des CNOTs aléatoires
        pairs = list(range(n_qubits))
        np.random.shuffle(pairs)
        for j in range(0, n_qubits - 1, 2):
            if pairs[j] != pairs[j + 1]:
                layer_gates.append(('cnot', pairs[j], pairs[j + 1]))
        gate_structure.append(layer_gates)

    # Calculer le nombre de paramètres
    # Chaque couche : n_qubits paramètres pour les portes aléatoires + embedding initial
    n_params_per_layer = n_qubits
    total_params = (n_layers + 1) * n_qubits  # embedding + n_layers

    @qml.qnode(dev, diff_method="parameter-shift")
    def circuit(params):
        qml.AngleEmbedding(params[0], wires=range(n_qubits))
        idx = 1
        for l in range(n_layers):
            for gate in gate_structure[l]:
                if gate[0] == 'single':
                    _, gtype, wire = gate
                    if gtype == 'RX':
                        qml.RX(params[idx][wire], wires=wire)
                    elif gtype == 'RY':
                        qml.RY(params[idx][wire], wires=wire)
                    else:
                        qml.RZ(params[idx][wire], wires=wire)
                elif gate[0] == 'cnot':
                    _, ctrl, target = gate
                    qml.CNOT(wires=[ctrl, target])
            idx += 1
        return qml.expval(qml.PauliZ(0))
    return circuit, total_params


def ansatz_gradient_variance(circuit_fn, n_qubits, n_layers, n_samples=60):
    """Variance du gradient pour un ansatz donné."""
    all_grads = []
    param_shape = (n_layers + 1, n_qubits)
    for _ in range(n_samples):
        params = np.random.uniform(0, 2 * np.pi, param_shape)
        grad = qml.grad(circuit_fn)(params)
        flat = np.array(grad, dtype=float).flatten()
        all_grads.extend(flat.tolist())
    return float(np.var(all_grads))

In [ ]:
n_range = [2, 4, 6, 8, 10]
var_struct, var_random = [], []

for n in n_range:
    c_struct = build_structured_ansatz(n, n_layers=2)
    c_rand, _ = build_random_ansatz(n, n_layers=2, seed=42)
    v_s = ansatz_gradient_variance(c_struct, n, n_layers=2, n_samples=50)
    v_r = ansatz_gradient_variance(c_rand, n, n_layers=2, n_samples=50)
    var_struct.append(v_s)
    var_random.append(v_r)
    print(f"n={n:2d}  structuré={v_s:.3e}  aléatoire={v_r:.3e}")

plt.figure(figsize=(8, 5))
plt.semilogy(n_range, var_struct, 'o-', label='Structuré (BasicEntanglerLayers)', lw=2)
plt.semilogy(n_range, var_random, 's--', label='Aléatoire (gates aléatoires)', lw=2)
plt.xlabel('Nombre de qubits $n$')
plt.ylabel('Var[grad]')
plt.title('Comparaison ansatz : structuré vs. aléatoire')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Analyse :** L'ansatz structuré (`BasicEntanglerLayers`) maintient une variance de gradient plus élevée que l'ansatz aléatoire, ce qui le rend préférable pour l'entraînement. La structure régulière des portes d'intrication et des rotations crée un paysage de coût plus favorable.

---
## Partie C : Coût global vs. local

Un résultat clé de Cerezo et al. (2021) : les fonctions de coût *locales* (qui n'impliquent qu'un sous-ensemble de qubits) ont une variance de gradient qui décroît moins vite que les coûts *globaux* (qui impliquent tous les qubits).

In [ ]:
def build_cost_comparison(n_qubits):
    """Crée deux QNodes : coût global et coût local."""
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, diff_method="parameter-shift")
    def circuit_global(params):
        qml.AngleEmbedding(params[0], wires=range(n_qubits))
        qml.BasicEntanglerLayers(params[1:], wires=range(n_qubits))
        # Opérateur global : produit tensoriel de PauliZ sur tous les qubits
        obs = qml.PauliZ(0)
        for i in range(1, n_qubits):
            obs = obs @ qml.PauliZ(i)
        return qml.expval(obs)

    @qml.qnode(dev, diff_method="parameter-shift")
    def circuit_local(params):
        qml.AngleEmbedding(params[0], wires=range(n_qubits))
        qml.BasicEntanglerLayers(params[1:], wires=range(n_qubits))
        # Opérateur local : PauliZ sur un seul qubit
        return qml.expval(qml.PauliZ(0))

    return circuit_global, circuit_local


def compute_cost_variance(qnode, n_qubits, n_layers, n_samples=60):
    """Variance du gradient pour un QNode donné."""
    all_grads = []
    param_shape = (n_layers + 1, n_qubits)
    for _ in range(n_samples):
        params = np.random.uniform(0, 2 * np.pi, param_shape)
        grad = qml.grad(qnode)(params)
        flat = np.array(grad, dtype=float).flatten()
        all_grads.extend(flat.tolist())
    return float(np.var(all_grads))

In [ ]:
n_range = [2, 3, 4, 5, 6, 7, 8, 9, 10]
var_global, var_local = [], []

for n in n_range:
    c_global, c_local = build_cost_comparison(n)
    vg = compute_cost_variance(c_global, n, n_layers=2, n_samples=60)
    vl = compute_cost_variance(c_local, n, n_layers=2, n_samples=60)
    var_global.append(vg)
    var_local.append(vl)
    print(f"n={n:2d}  global={vg:.3e}  local={vl:.3e}  rapport={vl/vg if vg>0 else float('inf'):.1f}")

plt.figure(figsize=(8, 5))
plt.semilogy(n_range, var_global, 'o-', label='Coût global (⨂ Z)', lw=2, color='#E74C3C')
plt.semilogy(n_range, var_local, 's--', label='Coût local (Z₀)', lw=2, color='#2ECC71')
plt.xlabel('Nombre de qubits $n$')
plt.ylabel('Var[grad]')
plt.title('Comparaison coût global vs. local')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Analyse :** Le coût local ($Z_0$ seul) présente une variance de gradient significativement plus élevée que le coût global ($\bigotimes Z$), surtout pour $n \geq 6$. C'est pourquoi les architectures QML modernes privilégient des observables locaux pour l'entraînement.

---
## Partie D : Stratégies d'initialisation

L'initialisation des paramètres a un impact direct sur la variance du gradient en début d'entraînement. Nous comparons trois stratégies :
- **Uniforme** : $\theta \sim \mathcal{U}(0, 2\pi)$ (défaut)
- **Normale** : $\theta \sim \mathcal{N}(0, \pi)$
- **Identité** : $\theta \sim \mathcal{N}(0, 0.1)$ (proche de l'identité, circuit presque trivial)

In [ ]:
def init_strategy_gradient_variance(n_qubits, n_layers=2, n_samples=60, strategy='uniform'):
    """Variance du gradient selon la stratégie d'initialisation."""
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, diff_method="parameter-shift")
    def circuit(params):
        qml.AngleEmbedding(params[0], wires=range(n_qubits))
        qml.BasicEntanglerLayers(params[1:], wires=range(n_qubits))
        return qml.expval(qml.PauliZ(0))

    all_grads = []
    param_shape = (n_layers + 1, n_qubits)
    for _ in range(n_samples):
        if strategy == 'uniform':
            params = np.random.uniform(0, 2 * np.pi, param_shape)
        elif strategy == 'normal':
            params = np.random.normal(0, np.pi, param_shape)
        elif strategy == 'identity':
            params = np.random.normal(0, 0.1, param_shape)
        else:
            raise ValueError(f"Stratégie inconnue : {strategy}")
        grad = qml.grad(circuit)(params)
        flat = np.array(grad, dtype=float).flatten()
        all_grads.extend(flat.tolist())
    return float(np.var(all_grads))

In [ ]:
strategies = ['uniform', 'normal', 'identity']
colors = {'uniform': '#E74C3C', 'normal': '#F39C12', 'identity': '#2ECC71'}
markers = {'uniform': 'o', 'normal': 's', 'identity': '^'}

n_range = [2, 4, 6, 8, 10]
results = {s: [] for s in strategies}

for n in n_range:
    for s in strategies:
        v = init_strategy_gradient_variance(n, n_layers=2, n_samples=50, strategy=s)
        results[s].append(v)
    print(f"n={n:2d}  uniforme={results['uniform'][-1]:.3e}  normale={results['normal'][-1]:.3e}  identité={results['identity'][-1]:.3e}")

plt.figure(figsize=(8, 5))
for s in strategies:
    plt.semilogy(n_range, results[s], f'{markers[s]}-', label=s, color=colors[s], lw=2)
plt.xlabel('Nombre de qubits $n$')
plt.ylabel('Var[grad]')
plt.title('Impact de la stratégie d\'initialisation sur la variance du gradient')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Analyse :** L'initialisation *identité* (paramètres proches de zéro) préserve une variance de gradient élevée, ce qui facilite l'apprentissage. En revanche, l'initialisation uniforme $\mathcal{U}(0, 2\pi)$ souffre du barren plateau dès $n \geq 6$. C'est une stratégie pratique simple pour atténuer le problème.

---
## Partie E : Visualisation avancée

Nous visualisons les distributions de gradients sous forme de boxplots et de heatmaps pour mieux comprendre la structure du paysage de coût.

In [ ]:
def collect_gradients(n_qubits, n_layers=2, n_samples=100):
    """Collecte les gradients pour plusieurs initialisations."""
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, diff_method="parameter-shift")
    def circuit(params):
        qml.AngleEmbedding(params[0], wires=range(n_qubits))
        qml.BasicEntanglerLayers(params[1:], wires=range(n_qubits))
        return qml.expval(qml.PauliZ(0))

    all_grads = []
    param_shape = (n_layers + 1, n_qubits)
    for _ in range(n_samples):
        params = np.random.uniform(0, 2 * np.pi, param_shape)
        grad = qml.grad(circuit)(params)
        flat = np.array(grad, dtype=float).flatten()
        all_grads.append(flat)
    return np.array(all_grads)


# Boxplots : distribution des gradients pour différentes tailles de système
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, n in enumerate([4, 6, 8]):
    grads = collect_gradients(n, n_layers=2, n_samples=80)
    axes[idx].boxplot(grads, vert=True, showfliers=False, patch_artist=True,
                      boxprops=dict(facecolor='#3498DB', alpha=0.6))
    axes[idx].set_title(f'n = {n} qubits')
    axes[idx].set_xlabel('Indice du paramètre')
    axes[idx].set_ylabel('Valeur du gradient')
    axes[idx].axhline(y=0, color='red', ls='--', alpha=0.5)
    axes[idx].grid(alpha=0.3)

plt.suptitle('Boxplots des gradients pour différentes tailles de système', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de la matrice de covariance des gradients
from matplotlib.colors import Normalize

n_qubits = 8
n_samples = 200
grads = collect_gradients(n_qubits, n_layers=2, n_samples=n_samples)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Heatmap 1 : matrice de covariance
cov_matrix = np.cov(grads.T)
im1 = axes[0].imshow(np.abs(cov_matrix), cmap='YlOrRd', aspect='auto',
                      norm=Normalize(vmin=0, vmax=np.max(np.abs(cov_matrix))))
axes[0].set_title('Matrice de covariance des gradients (|Cov|)')
axes[0].set_xlabel('Paramètre j')
axes[0].set_ylabel('Paramètre i')
plt.colorbar(im1, ax=axes[0])

# Heatmap 2 : corrélation
corr_matrix = np.corrcoef(grads.T)
# Masquer la diagonale pour mieux voir les corrélations croisées
masked_corr = np.where(np.abs(corr_matrix) < 0.1, 0, corr_matrix)
im2 = axes[1].imshow(masked_corr, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
axes[1].set_title('Matrice de corrélation des gradients')
axes[1].set_xlabel('Paramètre j')
axes[1].set_ylabel('Paramètre i')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print(f"Variance moyenne des gradients : {np.mean(np.var(grads, axis=0)):.3e}")
print(f"Corrélation moyenne (hors diagonale) : {np.mean(np.abs(corr_matrix[~np.eye(corr_matrix.shape[0], dtype=bool)])):.3e}")

**Analyse :** Les boxplots montrent que la dispersion des gradients se réduit considérablement quand $n$ augmente. La heatmap de covariance révèle la structure des corrélations entre paramètres — plus le circuit est profond, plus ces corrélations sont complexes.

---
## Exercices

1. **Profondeur du circuit :** Reproduisez la Partie A avec $L = 4, 6, 8$ couches. La variance diminue-t-elle avec la profondeur ?

2. **Mesure locale sur plusieurs qubits :** Comparez la variance pour $O = Z_0$, $O = Z_0 + Z_1$, $O = Z_0 Z_1$. La variance augmente-t-elle avec le nombre de termes locaux ?

3. **Initialisation identité avec bruit :** Testez l'initialisation $\theta \sim \mathcal{N}(0, \sigma)$ pour $\sigma \in \{0.01, 0.1, 0.5, 1.0\}$. Quel est l'impact de $\sigma$ sur la variance initiale et sur l'entraînement ?

4. **Ansatz avec coût local :** Combinez un coût local avec une initialisation identité. La variance est-elle préservée même pour $n = 12$ qubits ?

5. **Qubits de bruit :** Ajoutez 2 qubits auxiliaires qui ne sont pas mesurés. Le barren plateau s'aggrave-t-il ? (Indice : oui, car la trace partielle ajoute de l'aléatoire.)

6. **(Avancé) Parameter-shift vs. adjoint :** Comparez `diff_method="parameter-shift"` et `diff_method="adjoint"` pour le calcul du gradient. Quel est le compromis précision / vitesse ?